## Unit 3: Spatial Data

### Learning objectives
This module provides an introduction to the fundamentals of working with spatial vector and raster data in python while empirically exploring why systematic and structural racism is interwined with urban ecological processes.




### Background 

In August 2020, [Christopher Schell](http://directory.tacoma.uw.edu/employee/cjschell) and collegues published a review in *Science* on ['The ecological and evolutionary consequences of systemic racism in urban environments'](https://science.sciencemag.org/content/early/2020/08/12/science.aay4497) (DOI: 10.1126/science.aay4497), showing how systematic racism and classism  has significant impacts on ecological and evolutionary processes within urban environments. Here we explore a subset of the data used to support these findings in this review and the broader literature.



The [press release](https://www.washington.edu/news/2020/08/13/systemic-racism-has-consequences-for-all-life-in-cities/) on the paper is worth a read:

> “Racism is destroying our planet, and how we treat each other is essentially structural violence against our natural world,” said lead author Christopher Schell, an assistant professor of urban ecology at the University of Washington Tacoma. “Rather than just changing the conversation about how we treat each other, this paper will hopefully change the conversation about how we treat the natural world.”

We are going to explore one metric for how structural racism and classism underpin landscape heterogeneity in cities.



**Figure 2** in the Schell paper shows how NDVI (Normalized Difference Vegetation Index) tracks historical redlining.

```{image} attachment:23647e95-a23c-4eb8-a9bb-f0c91e45c1ac.png
:width: 300px
:align: center
```


We are going to recreate some of these city maps, and plot the distributions and mean vegetation patterns across cities to explore the structural inequality and racism that Schell et al highlight in their paper.


To do this we are going to use the following spatial data:  

**1. Mapping Inequality:** (vector data)  
Please take the time to read the introduction to this dataset [here](https://dsl.richmond.edu/panorama/redlining/#loc=3/41.245/-105.469&text=intro)

**2.  Satellite Imagery from Sentinel-2 catalog:** (raster data), from [Planetary Computer](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a)


## Vector data: the HOLC maps

Mapping Inequality project converted the 1930s Home Owners’ Loan Corporation maps into polygons, one each graded neighborhood, with its grade (A “best” to D “hazardous”), city, and original map color in the `fill` column. I read the GeoPackage via the web with `/vsicurl/` which only requests the bytes a query needs instead of downloading the complete file. ibis’ `read_geo` function raised a `KeyError: 'EPSG:4326'` because the version of ibis doesn’t understand the coordinate system label that DuckDB adds to the geometry column. So we read the table through a DuckDB view that converts the geometry to plain binary (WKB), and ibis filters and counts from there. I kept only the residential neighborhoods in New Haven, Oakland and Baltimore, because non-residential regions show up on the maps but were never assessed.

**Prompt to Kilo (code mode):**

```
In notebook.ipynb, add new cells at the end (don't change the existing ones). Use ibis with the DuckDB backend and the spatial extension to read the Mapping Inequality HOLC polygons from https://dsl.richmond.edu/panorama/redlining/static/mappinginequality.gpkg without downloading the whole file. Show the columns, check that "New Haven", "Oakland" and "Baltimore" appear as city names, filter to residential neighborhoods in those three cities, count neighborhoods by city and grade, convert the result to a geopandas GeoDataFrame, and draw New Haven on an interactive leafmap.maplibregl map with each neighborhood colored by its HOLC grade. One step per cell, no print statements, no comments, no try/except. Tell me your plan before editing anything.

Follow-up: Use con.read_geo with the /vsicurl/ URL directly, as plain ibis. Only switch to a SQL view if read_geo actually fails, and if it does, show me the exact error first. For the map, color each polygon with the existing "fill" column instead of a match expression.
```



In [1]:
import geopandas as gpd
import ibis
import leafmap.maplibregl as maplibregl

con = ibis.duckdb.connect()
con.load_extension("spatial")

In [2]:
con.raw_sql(
    """
    CREATE OR REPLACE VIEW holc AS
    SELECT * EXCLUDE (geom), ST_AsWKB(geom) AS geom
    FROM ST_Read('/vsicurl/https://dsl.richmond.edu/panorama/redlining/static/mappinginequality.gpkg')
    """
)
holc = con.table("holc")

In [3]:
holc.columns

('fid',
 'area_id',
 'city',
 'state',
 'city_survey',
 'category',
 'grade',
 'label',
 'residential',
 'commercial',
 'industrial',
 'fill',
 'geom')

In [4]:
cities = ["New Haven", "Oakland", "Baltimore"]
holc.filter(holc.city.isin(cities)).select("city").distinct().order_by("city").execute()

,city
0,Baltimore
1,New Haven
2,Oakland


In [5]:
neighborhoods = holc.filter(holc.residential & holc.city.isin(cities))
neighborhoods

r0 := DatabaseTable: holc
  fid         int64
  area_id     int32
  city        string
  state       string
  city_survey boolean
  category    string
  grade       string
  label       string
  residential boolean
  commercial  boolean
  industrial  boolean
  fill        string
  geom        binary

Filter[r0]
  r0.residential
  InValues(value=r0.city, options=['New Haven', 'Oakland', 'Baltimore'])

In [6]:
counts = (
    neighborhoods.group_by(["city", "grade"])
    .agg(neighborhoods.fid.count().name("neighborhood_count"))
    .order_by(["city", "grade"])
)
counts.execute()

,city,grade,neighborhood_count
0,Baltimore,A,6
1,Baltimore,B,22
2,Baltimore,C,22
3,Baltimore,D,9
4,New Haven,A,8
5,New Haven,B,10
6,New Haven,C,20
7,New Haven,D,9
8,Oakland,A,12
9,Oakland,B,46


In [7]:
neighborhoods_df = neighborhoods.to_pandas()
gdf = gpd.GeoDataFrame(
    neighborhoods_df.drop(columns="geom"),
    geometry=gpd.GeoSeries.from_wkb(neighborhoods_df["geom"], crs="EPSG:4326"),
)
gdf

,fid,area_id,city,state,city_survey,category,grade,label,residential,commercial,industrial,fill,geometry
0,659,6690,Oakland,CA,True,Best,A,A,True,False,False,#76a865,"MULTIPOLYGON (((-122.20252 37.81099, -122.2021..."
1,660,6662,Oakland,CA,True,Best,A,A1,True,False,False,#76a865,"MULTIPOLYGON (((-122.25734 37.89463, -122.2580..."
2,661,6710,Oakland,CA,True,Best,A,A10,True,False,False,#76a865,"MULTIPOLYGON (((-122.23403 37.80671, -122.2350..."
3,662,6691,Oakland,CA,True,Best,A,A11,True,False,False,#76a865,"MULTIPOLYGON (((-122.20252 37.81099, -122.2023..."
4,663,6663,Oakland,CA,True,Best,A,A2,True,False,False,#76a865,"MULTIPOLYGON (((-122.27847 37.8971, -122.27844..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,4155,2891,Baltimore,MD,True,Hazardous,D,D4,True,False,False,#d9838d,"MULTIPOLYGON (((-76.60837 39.30615, -76.60928 ..."
221,4156,2911,Baltimore,MD,True,Hazardous,D,D5,True,False,False,#d9838d,"MULTIPOLYGON (((-76.63004 39.28575, -76.63106 ..."
222,4157,2873,Baltimore,MD,True,Hazardous,D,D6,True,False,False,#d9838d,"MULTIPOLYGON (((-76.61981 39.3189, -76.61997 3..."
223,4158,7274,Baltimore,MD,True,Hazardous,D,D8,True,False,False,#d9838d,"MULTIPOLYGON (((-76.61815 39.31018, -76.61897 ..."


In [8]:
gdf.total_bounds

array([-122.3097 ,   37.71369,  -72.85227,   41.37502])

The bounds run from about −122.3 to −72.9 in longitude and 37.7 to 41.4 in latitude, which matches the locations of Oakland and New Haven, confirming that the polygons are in longitude/latitude (EPSG:4326) as declared.

In [9]:
new_haven = gdf[gdf["city"] == "New Haven"]
new_haven

,fid,area_id,city,state,city_survey,category,grade,label,residential,commercial,industrial,fill,geometry
119,1306,3569,New Haven,CT,True,Best,A,A1,True,False,False,#76a865,"MULTIPOLYGON (((-72.9 41.36885, -72.90325 41.3..."
120,1307,3568,New Haven,CT,True,Best,A,A2,True,False,False,#76a865,"MULTIPOLYGON (((-72.89401 41.36331, -72.89256 ..."
121,1308,3566,New Haven,CT,True,Best,A,A3,True,False,False,#76a865,"MULTIPOLYGON (((-72.90981 41.35949, -72.90284 ..."
122,1309,3567,New Haven,CT,True,Best,A,A4,True,False,False,#76a865,"MULTIPOLYGON (((-72.90284 41.35795, -72.8999 4..."
123,1310,3564,New Haven,CT,True,Best,A,A5,True,False,False,#76a865,"MULTIPOLYGON (((-72.89801 41.3493, -72.89883 4..."
124,1311,3557,New Haven,CT,True,Best,A,A6,True,False,False,#76a865,"MULTIPOLYGON (((-72.91231 41.33334, -72.91188 ..."
125,1312,3546,New Haven,CT,True,Best,A,A7,True,False,False,#76a865,"MULTIPOLYGON (((-72.97003 41.32192, -72.97004 ..."
126,1313,3556,New Haven,CT,True,Best,A,A8,True,False,False,#76a865,"MULTIPOLYGON (((-72.91361 41.33346, -72.91428 ..."
127,1314,3570,New Haven,CT,True,Still Desirable,B,B1,True,False,False,#7cb5bd,"MULTIPOLYGON (((-72.9 41.36885, -72.89688 41.3..."
128,1315,3576,New Haven,CT,True,Still Desirable,B,B10,True,False,False,#7cb5bd,"MULTIPOLYGON (((-72.86698 41.2815, -72.86971 4..."


In [10]:
nh = new_haven[['grade', 'label', 'fill', 'geometry']]
m = maplibregl.Map(
    center=[-72.93, 41.31], 
    zoom=12, 
    style="positron", 
    add_sidebar=False, 
    add_floating_sidebar=False
)
m.add_gdf(
    nh, 
    layer_type="fill", 
    paint={
        "fill-color": ["get", "fill"], 
        "fill-opacity": 0.65
    }, 
    name="New Haven HOLC grades")

In [11]:
def save_map(gdf_in, path, center, zoom):
    page = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<style>body {{margin: 0}} #map {{position: absolute; top: 0; bottom: 0; width: 100%}}</style>
</head><body><div id="map"></div><script>
const map = new maplibregl.Map({{container: 'map', style: 'https://basemaps.cartocdn.com/gl/positron-gl-style/style.json', center: {center}, zoom: {zoom}}});
map.on('load', () => {{
  map.addSource('holc', {{type: 'geojson', data: {gdf_in.to_json()}}});
  map.addLayer({{id: 'holc', type: 'fill', source: 'holc', paint: {{'fill-color': ['get', 'fill'], 'fill-opacity': 0.65, 'fill-outline-color': '#ffffff'}}}});
}});
</script></body></html>"""
    open(path, "w").write(page)

save_map(nh, "new_haven_holc.html", [-72.93, 41.31], 12)

leafmap's HTML export wrote the neighborhood shapes into the file but did not draw them, even with default styling, so I write the HTML page directly with MapLibre, the same mapping library leafmap uses, embedding the polygons as GeoJSON.

The map shows New Haven's residential neighborhoods in their original HOLC colors: green for A, blue for B, yellow for C and red for D.

**Prompt to Kilo (code mode):**

```
In notebook.ipynb, add new cells after the existing ones. For each of the three cities in gdf, use pystac_client to search the Earth Search STAC catalog (https://earth-search.aws.element84.com/v1, collection sentinel-2-l2a) over that city's bounding box for June 1 to September 1, 2024, with cloud cover under 20%. Load only the red and nir08 bands lazily with odc.stac (chunks={}, groupby="solar_day", 10 m resolution), compute NDVI as (nir08 - red) / (nir08 + red), take the median over time, and save each city's NDVI as a Cloud-Optimized GeoTIFF reprojected to EPSG:4326. Then show New Haven's NDVI on a leafmap.maplibregl map with a green color scale and the HOLC polygons on top. One step per cell, no print statements, no comments, no try/except; a small function is fine only if it avoids repeating the same code for each city. Tell me your plan before editing anything.
```

In [12]:
from pathlib import Path

from pystac_client import Client
import odc.stac
import rioxarray

ndvi_dir = Path("ndvi")
ndvi_dir.mkdir(exist_ok=True)

In [13]:
client = Client.open("https://earth-search.aws.element84.com/v1")

In [14]:
city_bboxes = {city: tuple(gdf.loc[gdf["city"] == city].total_bounds) for city in cities}
city_bboxes

{'New Haven': (np.float64(-72.99321),
  np.float64(41.23109),
  np.float64(-72.85227),
  np.float64(41.37502)),
 'Oakland': (np.float64(-122.3097),
  np.float64(37.71369),
  np.float64(-122.11685),
  np.float64(37.90601)),
 'Baltimore': (np.float64(-76.7569),
  np.float64(39.17875),
  np.float64(-76.50927),
  np.float64(39.4068))}

In [15]:
items_by_city = {
    city: list(
        client.search(
            collections=["sentinel-2-l2a"],
            bbox=bbox,
            datetime="2024-06-01/2024-09-01",
            query={"eo:cloud_cover": {"lt": 20}},
        ).items()
    )
    for city, bbox in city_bboxes.items()
}
{city: len(items) for city, items in items_by_city.items()}

{'New Haven': 4, 'Oakland': 18, 'Baltimore': 15}

In [16]:
items_by_city["New Haven"][0].assets["red"].extra_fields.get("raster:bands")

[{'nodata': 0,
  'data_type': 'uint16',
  'bits_per_sample': 15,
  'spatial_resolution': 10,
  'scale': 0.0001,
  'offset': -0.1}]

In [23]:
{(i.assets['red'].extra_fields['raster:bands'][0]['offset'], i.assets['nir08'].extra_fields['raster:bands'][0]['offset']) for its in items_by_city.values() for i in its}

{(-0.1, -0.1)}

In [30]:
def save_city_ndvi(city, bbox, items):
    ds = odc.stac.load(
        items,
        bands=["red", "nir08"],
        bbox=bbox,
        crs="utm",
        resolution=30,
        chunks={},
        groupby="solar_day",
    )
    nir = ds.nir08.where(ds.nir08 != 0).astype("float32") * 0.0001 - 0.1
    red = ds.red.where(ds.red != 0).astype("float32") * 0.0001 - 0.1
    ndvi = ((nir - red) / (nir + red)).where((nir > 0) & (red > 0))
    ndvi_median = ndvi.median("time")
    ndvi_4326 = (
        ndvi_median.rio.write_crs(ds.odc.crs)
        .rio.write_nodata(float("nan"))
        .rio.reproject("EPSG:4326", nodata=float("nan"))
    )
    path = ndvi_dir / f"{city.lower().replace(' ', '_')}_ndvi_2024.tif"
    ndvi_4326.rio.to_raster(path, driver="COG", compress="DEFLATE", nodata=float("nan"))
    return ndvi_4326, str(path)

In [31]:
new_haven_ndvi, new_haven_ndvi_path = save_city_ndvi(
    "New Haven",
    city_bboxes["New Haven"],
    items_by_city["New Haven"],
)

In [32]:
oakland_ndvi, oakland_ndvi_path = save_city_ndvi(
    "Oakland",
    city_bboxes["Oakland"],
    items_by_city["Oakland"],
)

In [33]:
baltimore_ndvi, baltimore_ndvi_path = save_city_ndvi(
    "Baltimore",
    city_bboxes["Baltimore"],
    items_by_city["Baltimore"],
)

In [34]:
ndvi_stats = {
    city: {
        "min": float(ndvi.min()),
        "max": float(ndvi.max()),
        "mean": float(ndvi.mean()),
    }
    for city, ndvi in {
        "New Haven": new_haven_ndvi,
        "Oakland": oakland_ndvi,
        "Baltimore": baltimore_ndvi,
    }.items()
}
ndvi_stats

{'New Haven': {'min': -0.9933558702468872,
  'max': 0.9994584918022156,
  'mean': 0.32831287384033203},
 'Oakland': {'min': -0.9883041381835938,
  'max': 0.9995593428611755,
  'mean': 0.509689450263977},
 'Baltimore': {'min': -0.9838709831237793,
  'max': 0.9995410442352295,
  'mean': 0.5353455543518066}}

The catalog metadata for the red band includes a scale of 0.0001 and offset of −0.1, hence reflectance = stored value × 0.0001 − 0.1 . The same offset is used for all scenes of both bands. The original model version directly computes NDVI from the stored numbers; the offset cancels in the numerator but not the denominator itself, hence NDVI was distorted without error. After conversion to reflectance and treating negative reflectance (extremely dark water and shadow) as missing, mean NDVI throughout each city bounding box is 0.33 for New Haven, 0.51 for Oakland and 0.54 for Baltimore compared to 0.48, 0.31 and 0.48 before the modification. New Haven has the lowest average because its box includes a substantial portion of harbor and Long Island Sound, which has strongly negative NDVI water. I load the images at 30 m (not the native 10 m, as the 10m version stalled for Baltimore) yet 30m pixels are still much smaller than a neighborhood.

In [35]:
m = maplibregl.Map(
    center=[-72.93, 41.31],
    zoom=11,
    style="positron",
    add_sidebar=False,
    add_floating_sidebar=False,
)
m.add_raster(
    new_haven_ndvi_path,
    colormap="Greens",
    vmin=-1,
    vmax=1,
    nodata=float("nan"),
    name="New Haven NDVI",
    fit_bounds=True,
)
m.add_gdf(
    new_haven,
    layer_type="fill",
    paint={
        "fill-color": ["get", "fill"],
        "fill-opacity": 0.45,
        "fill-outline-color": "#ffffff",
    },
    name="New Haven HOLC grades",
    fit_bounds=False,
)
m